# Image Rotate


In [1]:
import os
os.chdir("computer-systems/bits-and-bytes/image-rotate")

The goal is to write a program that rotates a BMP image. Given `teapot.bmp` the output should match `rotated.bmp` exactly.

Let's start by inspecting the bytes and building an understanding of how the `bmp` format works.

In [2]:
with open("teapot.bmp", 'rb') as f:
    head = f.read(20)
    print(head)


b'BM\xba\x13\x08\x00\x00\x00\x00\x00\x8a\x00\x00\x00|\x00\x00\x00\xa4\x01'


Wikipedia has a great reference image: https://upload.wikimedia.org/wikipedia/commons/7/75/BMPfileFormat.svg. A BMP file is made up of:
1. Bitmap file header: signature, file size, file offset to pixel array
2. DIB header: image width and height, bits per pixel, etc.
3. Colour table
4. GAP1
5. Image data
6. GAP2
7. ICC Colour profile

All of the integer values are stored in little-endian format (I think).

Plan:
1. Parse the bitmap file header
2. Parse the DIB header
3. Transpose pixel data (this should achieve the 90 degree rotation we're looking for)

The file header is 14 bytes long and always begins with the characters "BM" in ASCII.

| Offset | Size | Purpose |
| --- | --- | --- |
| 00 | 2 bytes | Usually BM but other entries are possible |
| 02 | 4 bytes | The size of the BMP file in bytes |
| 06 | 2 bytes | Reserved |
| 08 | 2 bytes | Reserved |
| 0A | 4 bytes | The offset of the byte where the pixel data can be found |

In [3]:
def read_file(path: str) -> bytes:
    with open(path, "rb") as f:
        return f.read()

In [4]:
# struct provides functionality for converting between Python values and C-style structs
import struct

header_format = "<2cI2hI"
header_fields = ("B", "M", "size", "reserved", "reserved", "offset")

def parse_header(data: bytes):
    header = struct.unpack(header_format, data[0:struct.calcsize(header_format)])
    for field, value in zip(header, header_fields):
        print(f"{field}: {value}")

In [5]:
data = read_file("teapot.bmp")
parse_header(data)

b'B': B
b'M': M
529338: size
0: reserved
0: reserved
138: offset


The DIB header is more complex. It always starts with the size of the header, which can help determine which header is being used.

In [6]:
with open("teapot.bmp", 'rb') as f:
    f.seek(14)
    bytes = f.read(4)
    print(int.from_bytes(bytes, "little"))

124


This means that the image uses the `BITMAPV5HEADER` which adds ICC colour profiles.

In [13]:
dib_format = "<IiihhII"
dib_fields = ("header size", "bitmap width (px)", "bitmap height (px)", "colour planes (always 1)", "bits per pixel")

def parse_dib_header(data: bytes):
    header = struct.unpack(dib_format, data[14:14+struct.calcsize(dib_format)])
    for field, value in zip(dib_fields, header):
        print(f"{field}: {value}")

In [14]:
data = read_file("teapot.bmp")
parse_dib_header(data)

header size: 124
bitmap width (px): 420
bitmap height (px): 420
colour planes (always 1): 1
bits per pixel: 24
